# `03_r4s3_cop_consistency.ipynb` — R4-S3-COP **consistency check** (NOT a novel finding)

**Spec citations:** `docs/specs/2026-05-16-ai-cost-factor-model-design.md` v0.2.7 — §2.2.A (lines 1212–1234), §0.1 CORRECTIONS-S (R4-S3 split into COP-consistency vs USD-behavioral arms), §0.1 CORRECTIONS-I (HAC bandwidth = $\lfloor T^{1/3}\rfloor$ Andrews 1991), §0.3 CORRECTIONS-Y-6 ($\hat\pi$ ephemeral-cost disclosure), v0.2.7 Y-9 closure (ccusage parity = 1.000000×).

## CORRECTIONS-S preamble (pinned)

> **Confirming $\alpha_1^{COP} > 0$ here recovers a well-established USDCOP vol-clustering stylized fact, independent of AI-cost transmission. This arm has NO framework-graduation authority.** Under the subscription regime the NotionalCost$^{USD}$ side has bounded variation (token elasticity to FX is the open question, not the test content), so $|\Delta\ln \text{NotionalCost}^{COP}|$ is dominated by $|\Delta\ln \text{USDCOP}|$. The R4-S3-COP arm is included to **verify pipeline integrity** — that vol propagates correctly through the panel-builder.
>
> - **CONSISTENCY-PASS** = pipeline integrity verified (vol propagates correctly through the panel-builder); $\hat\alpha_1^{COP} > 0$ at $p_{\text{one-sided}} < 0.05$. NOT a novel finding. NOT framework-graduation.
> - **CONSISTENCY-FAIL** = HALT to disposition memo (suspect data corruption); NOT a framework rejection.

## v0.2.7 Y-9 closure note (pinned)

> The production panel `data/panels/notional_cost_panel.parquet` is post-PR#4 (HEAD `11e238d`) and is verified at **1.000000× ccusage parity** (UTC apples-to-apples — `notebook(dev_ai_cost_v2): 01_data_eda — trio + reconciliation + power HALT (spec v0.2.5 §4)` and follow-on Y-9 fixes). Therefore any CONSISTENCY-FAIL fired here **cannot be attributed to parser-divergence or cost-data-pipeline issues** and would necessarily indicate downstream data corruption — escalate to disposition memo, do NOT continue silently.

## $\hat\pi$ disclaimer (CORRECTIONS-Y-6 — pinned)

> Cost magnitudes throughout the panel apply the 5m-tier rate to total cache-creation tokens. True economic cost is up to $1 + \hat\pi\cdot(\text{rate}_{1h}/\text{rate}_{5m} - 1) \approx 1.40\times$ higher ($\hat\pi \approx 0.399$ from the panel). **The OLS coefficients $\hat\alpha_1^{COP}$ and the one-sided p-value are unaffected by this multiplicative bias** — the bias is a constant scaling of the LHS in level terms but cancels in the log-difference (a constant multiplier on the LHS adds a constant to $\ln\text{Cost}$ which differences away). The sign-and-significance verdict on $\alpha_1^{COP}$ is robust to the $\hat\pi$ disclosure.

## Anti-fishing pins (set PRE-DATA in Cell 2 — DO NOT TUNE)

| Pin | Value | Source |
|---|---|---|
| LAG_PRIMARY | `1` | §2.2.A — $k=1$ primary |
| LAG_SENSITIVITY | `5` | §2.2.A — $k=5$ sensitivity |
| HAC_LAGS | $\lfloor T_{\text{lag}}^{1/3}\rfloor$ | §0.1 CORRECTIONS-I (Andrews 1991) |
| ALPHA_LEVEL | `0.05` (one-sided) | §2.2.A verdict logic |
| Tokens-proxy | `output_tok` | largest token class — best proxy for token-volume |
| USDCOP series | `trm_cop_per_usd` | Banrep TRM |
| Verdict labels | `{CONSISTENCY-PASS, CONSISTENCY-FAIL}` only | §2.2.A — verdict-bearing for pipeline integrity only |

## Cell 2 — Anti-fishing pins (declared PRE-DATA)

In [1]:
from __future__ import annotations

import numpy as np

# ANTI-FISHING PINS — set PRE-DATA, DO NOT TUNE
LAG_PRIMARY = 1            # §2.2.A primary lag (k=1)
LAG_SENSITIVITY = 5        # §2.2.A sensitivity lag (k=5)
ALPHA_LEVEL = 0.05         # one-sided test threshold
TOKENS_PROXY = 'output_tok'  # largest token class; best proxy for token-volume
USDCOP_SERIES = 'trm_cop_per_usd'  # Banrep TRM
COST_SERIES = 'notional_cost_cop'

# HAC bandwidth pinned PRE-DATA as functional form floor(T^(1/3)) per Andrews 1991
def hac_bandwidth(t_lag: int) -> int:
    """Andrews (1991) rule-of-thumb HAC bandwidth: L = floor(T^(1/3))."""
    return int(np.floor(t_lag ** (1.0 / 3.0)))

VERDICT_LABELS = ('CONSISTENCY-PASS', 'CONSISTENCY-FAIL')

print('PINS (declared PRE-DATA, DO NOT TUNE):')
print(f'  LAG_PRIMARY      = {LAG_PRIMARY}')
print(f'  LAG_SENSITIVITY  = {LAG_SENSITIVITY}')
print(f'  HAC_LAGS         = floor(T_lag^(1/3))   [functional form]')
print(f'  ALPHA_LEVEL      = {ALPHA_LEVEL} (one-sided test)')
print(f'  TOKENS_PROXY     = {TOKENS_PROXY!r}')
print(f'  USDCOP_SERIES    = {USDCOP_SERIES!r}')
print(f'  COST_SERIES      = {COST_SERIES!r}')
print(f'  VERDICT_LABELS   = {VERDICT_LABELS}')

PINS (declared PRE-DATA, DO NOT TUNE):
  LAG_PRIMARY      = 1
  LAG_SENSITIVITY  = 5
  HAC_LAGS         = floor(T_lag^(1/3))   [functional form]
  ALPHA_LEVEL      = 0.05 (one-sided test)
  TOKENS_PROXY     = 'output_tok'
  USDCOP_SERIES    = 'trm_cop_per_usd'
  COST_SERIES      = 'notional_cost_cop'
  VERDICT_LABELS   = ('CONSISTENCY-PASS', 'CONSISTENCY-FAIL')


## Trio 1 — Construct $|\Delta\ln\text{NotionalCost}^{COP}|$, $|\Delta\ln\text{USDCOP}|$, $|\Delta\ln\text{Tokens}|$

### Decision-citation block (4-part)

- **Reference:** spec §2.2.A spec equation (lines 1215–1217).
- **Why:** absolute log-returns are the canonical vol-clustering input. The R4-S3 family regresses $|\Delta\ln\text{LHS}|_t$ on lagged $|\Delta\ln\text{RHS}|_{t-k}$ — a vol-on-vol specification (Engle 1982 ARCH framework). First-differencing the natural log enforces approximate stationarity; the absolute value extracts the magnitude of the daily move (the vol-clustering target).
- **Relevance:** Engle 1982 ARCH framework. Volatility clustering is the well-established stylized fact this arm recovers; the magnitudes are the regression inputs.
- **Connection:** feeds Trio 2 (OLS primary) and Trio 3 (OLS sensitivity). All downstream Newey-West HAC regressions consume `abs_dln_cop`, `abs_dln_trm`, `abs_dln_tok`.

### Why

Load the production panel and compute the absolute first-difference of the natural log of the three series: `notional_cost_cop` (LHS), `trm_cop_per_usd` (RHS₁, USDCOP), and `output_tok` (RHS₂, tokens-proxy).

In [2]:
from pathlib import Path

import polars as pl

PANEL_PATH = Path('../../data/panels/notional_cost_panel.parquet')
df = pl.read_parquet(PANEL_PATH)
print(f'panel shape: {df.shape}')
print(f'date window: {df["date_utc"].min()} -> {df["date_utc"].max()}')
print(f'columns: {df.columns}')

# Compute absolute first-diff of natural log for the three series
abs_dln_cop = df[COST_SERIES].log().diff().abs().drop_nulls().to_numpy()
abs_dln_trm = df[USDCOP_SERIES].log().diff().abs().drop_nulls().to_numpy()
abs_dln_tok = df[TOKENS_PROXY].log().diff().abs().drop_nulls().to_numpy()

T = len(abs_dln_cop)
assert len(abs_dln_trm) == T and len(abs_dln_tok) == T, 'series length mismatch after first-diff'

print(f'\nT after first-diff: {T}  (panel rows N = {df.shape[0]}, T = N - 1)')
print(f'NaN check — abs_dln_cop: {int(np.isnan(abs_dln_cop).sum())}, abs_dln_trm: {int(np.isnan(abs_dln_trm).sum())}, abs_dln_tok: {int(np.isnan(abs_dln_tok).sum())}')

print('\nRange / summary stats:')
for name, s in [('|Δln Cost^COP|', abs_dln_cop), ('|Δln USDCOP|', abs_dln_trm), ('|Δln Tokens|', abs_dln_tok)]:
    print(f'  {name:18s}: mean={s.mean():.6f}, sd={s.std(ddof=1):.6f}, min={s.min():.6f}, max={s.max():.6f}')

panel shape: (29, 11)
date window: 2026-01-06 -> 2026-05-14
columns: ['date_utc', 'notional_cost_usd', 'notional_cost_cop', 'trm_cop_per_usd', 'input_tok', 'output_tok', 'cache_create_5m', 'cache_create_1h', 'cache_read', 'n_messages', 'ephemeral_pi_share']

T after first-diff: 28  (panel rows N = 29, T = N - 1)
NaN check — abs_dln_cop: 0, abs_dln_trm: 0, abs_dln_tok: 0

Range / summary stats:
  |Δln Cost^COP|    : mean=1.167342, sd=1.152415, min=0.002468, max=5.478575
  |Δln USDCOP|      : mean=0.007478, sd=0.004440, min=0.000632, max=0.023392
  |Δln Tokens|      : mean=1.442285, sd=1.689745, min=0.000000, max=7.697189


### Interpretation

$T$ is the post-first-difference series length (panel $N=29$ → 28 differences). All three series are NaN-free and finite. The range/summary stats expose the order-of-magnitude difference between the bursty single-developer usage series (`|Δln Cost^COP|`, `|Δln Tokens|`) and the small-magnitude FX series (`|Δln USDCOP|`) — the FX series will need to do real work via its coefficient $\hat\alpha_1^{COP}$ to materially explain LHS variation. This is exactly the spec's expectation: under subscription regime, the LHS is dominated by the additive identity $\Delta\ln\text{Cost}^{COP} = \Delta\ln\text{Cost}^{USD} + \Delta\ln\text{USDCOP}$, so the FX term enters with $\alpha_1^{COP} > 0$ by construction.

## Trio 2 — OLS $k=1$ primary with Newey-West HAC SE

### Decision-citation block (4-part)

- **Reference:** spec §2.2.A + §0.1 CORRECTIONS-I (HAC bandwidth rule).
- **Why:** HAC bandwidth $\lfloor T^{1/3}\rfloor$ per Andrews (1991) rule-of-thumb — the spec-pinned bandwidth formula. Newey-West HAC SE is the canonical heteroskedasticity-and-autocorrelation-robust covariance estimator for short-T regression with potentially serially correlated residuals; required because the absolute log-return process has well-known persistence (vol clustering itself).
- **Relevance:** one-sided test on $\alpha_1^{COP} > 0$ is pre-pinned in §2.2.A (lines 1219–1220). The verdict reduces to a one-sided p-value on the FX coefficient.
- **Connection:** primary regression for the CONSISTENCY-PASS / CONSISTENCY-FAIL verdict. Trio 3's $k=5$ sensitivity provides corroboration but does NOT override the primary.

### Why

Regress $|\Delta\ln\text{Cost}^{COP}|_t$ on a constant + $|\Delta\ln\text{USDCOP}|_{t-1}$ + $|\Delta\ln\text{Tokens}|_{t-1}$ with Newey-West HAC SE at bandwidth $L = \lfloor T_{\text{lag}}^{1/3}\rfloor$, and report the one-sided p-value on $\alpha_1^{COP}$.

In [3]:
import statsmodels.api as sm
from scipy.stats import t as student_t

# Lag-aligned arrays for k=1 primary
T_lag_primary = T - LAG_PRIMARY
hac_L_primary = hac_bandwidth(T_lag_primary)
print(f'T_lag (k={LAG_PRIMARY}) = {T_lag_primary}')
print(f'HAC L  = floor(T_lag^(1/3)) = floor({T_lag_primary**(1/3):.4f}) = {hac_L_primary}')

y_primary = abs_dln_cop[LAG_PRIMARY:]                # LHS at time t  (length T_lag)
x1_primary = abs_dln_trm[:-LAG_PRIMARY]              # |Δln USDCOP|_{t-1}
x2_primary = abs_dln_tok[:-LAG_PRIMARY]              # |Δln Tokens|_{t-1}
assert len(y_primary) == len(x1_primary) == len(x2_primary) == T_lag_primary, 'alignment broken'

X_primary = sm.add_constant(np.column_stack([x1_primary, x2_primary]))
model_primary = sm.OLS(y_primary, X_primary).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_primary})
print('\n' + str(model_primary.summary()))

# Extract α₁^COP (coefficient on |Δln USDCOP|_{t-1}, index 1 in [const, x1, x2])
alpha_1_cop_primary = float(model_primary.params[1])
se_alpha_1_primary = float(model_primary.bse[1])
t_stat_primary = alpha_1_cop_primary / se_alpha_1_primary
df_resid_primary = T_lag_primary - X_primary.shape[1]
p_one_sided_primary = float(1.0 - student_t.cdf(t_stat_primary, df=df_resid_primary))
verdict_primary = 'CONSISTENCY-PASS' if (alpha_1_cop_primary > 0 and p_one_sided_primary < ALPHA_LEVEL) else 'CONSISTENCY-FAIL'

print('\n' + '-' * 60)
print(f'k={LAG_PRIMARY} PRIMARY  (df_resid = {df_resid_primary}, HAC L = {hac_L_primary})')
print('-' * 60)
print(f'  α̂₁^COP             = {alpha_1_cop_primary:+.6f}')
print(f'  HAC SE             = {se_alpha_1_primary:.6f}')
print(f'  t-stat             = {t_stat_primary:+.4f}')
print(f'  p_one_sided (α₁>0) = {p_one_sided_primary:.6f}')
print(f'  vs ALPHA_LEVEL     = {ALPHA_LEVEL}')
print(f'  → verdict (primary) = {verdict_primary}')

T_lag (k=1) = 27
HAC L  = floor(T_lag^(1/3)) = floor(3.0000) = 3

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                 -0.076
Method:                 Least Squares   F-statistic:                    0.2503
Date:                Sun, 17 May 2026   Prob (F-statistic):              0.781
Time:                        23:21:07   Log-Likelihood:                -41.618
No. Observations:                  27   AIC:                             89.24
Df Residuals:                      24   BIC:                             93.12
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------

### Interpretation

$\hat\alpha_1^{COP}$ is the point estimate of the FX-vol coefficient at $k=1$. Under the additive identity (subscription regime + bounded USD-side variation), this coefficient should be positive and statistically significant by construction — vol propagation through the panel-builder simply re-exposes the USDCOP magnitude inside the LHS. A one-sided $p<0.05$ with $\hat\alpha_1^{COP}>0$ ⇒ **CONSISTENCY-PASS** (pipeline integrity verified, NOT a novel finding). Any other outcome ⇒ **CONSISTENCY-FAIL** ⇒ HALT to disposition memo (suspect data corruption). The Y-9 closure note means a FAIL cannot be parser-divergence.

## Trio 3 — $k=5$ sensitivity (weekly-horizon robustness)

### Decision-citation block (4-part)

- **Reference:** spec §2.2.A $k=5$ sensitivity (line 1219).
- **Why:** weekly-horizon robustness check. Vol-clustering effects can attenuate or persist over multi-day windows; a $k=5$ lag exposes whether the consistency property holds at the trading-week horizon as well as one day.
- **Relevance:** anti-fishing — both lag specifications were pinned PRE-DATA in Cell 2. The sensitivity is corroborative; the **primary verdict remains the $k=1$ result** regardless of the $k=5$ outcome. If $k=1$ and $k=5$ disagree, the primary verdict still governs and the disagreement is documented in the headline.
- **Connection:** uses the same Newey-West HAC machinery with bandwidth re-derived from the (smaller) $T_{\text{lag}}$ at $k=5$; same OLS / one-sided test geometry as Trio 2.

### Why

Same regression as Trio 2 with `LAG_SENSITIVITY = 5` in place of `LAG_PRIMARY = 1`. Report $\hat\alpha_1^{COP}$, HAC SE, $t$-stat, one-sided $p$-value, and compare to the primary result.

In [4]:
T_lag_sens = T - LAG_SENSITIVITY
hac_L_sens = hac_bandwidth(T_lag_sens)
print(f'T_lag (k={LAG_SENSITIVITY}) = {T_lag_sens}')
print(f'HAC L  = floor(T_lag^(1/3)) = floor({T_lag_sens**(1/3):.4f}) = {hac_L_sens}')

y_sens = abs_dln_cop[LAG_SENSITIVITY:]
x1_sens = abs_dln_trm[:-LAG_SENSITIVITY]
x2_sens = abs_dln_tok[:-LAG_SENSITIVITY]
assert len(y_sens) == len(x1_sens) == len(x2_sens) == T_lag_sens, 'alignment broken'

X_sens = sm.add_constant(np.column_stack([x1_sens, x2_sens]))
model_sens = sm.OLS(y_sens, X_sens).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_sens})
print('\n' + str(model_sens.summary()))

alpha_1_cop_sens = float(model_sens.params[1])
se_alpha_1_sens = float(model_sens.bse[1])
t_stat_sens = alpha_1_cop_sens / se_alpha_1_sens
df_resid_sens = T_lag_sens - X_sens.shape[1]
p_one_sided_sens = float(1.0 - student_t.cdf(t_stat_sens, df=df_resid_sens))
verdict_sens = 'CONSISTENCY-PASS' if (alpha_1_cop_sens > 0 and p_one_sided_sens < ALPHA_LEVEL) else 'CONSISTENCY-FAIL'

print('\n' + '-' * 60)
print(f'k={LAG_SENSITIVITY} SENSITIVITY  (df_resid = {df_resid_sens}, HAC L = {hac_L_sens})')
print('-' * 60)
print(f'  α̂₁^COP             = {alpha_1_cop_sens:+.6f}')
print(f'  HAC SE             = {se_alpha_1_sens:.6f}')
print(f'  t-stat             = {t_stat_sens:+.4f}')
print(f'  p_one_sided (α₁>0) = {p_one_sided_sens:.6f}')
print(f'  vs ALPHA_LEVEL     = {ALPHA_LEVEL}')
print(f'  → verdict (sensit) = {verdict_sens}')

T_lag (k=5) = 23
HAC L  = floor(T_lag^(1/3)) = floor(2.8439) = 2

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.092
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     1.366
Date:                Sun, 17 May 2026   Prob (F-statistic):              0.278
Time:                        23:21:07   Log-Likelihood:                -35.742
No. Observations:                  23   AIC:                             77.48
Df Residuals:                      20   BIC:                             80.89
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------

### Interpretation

The $k=5$ sensitivity at weekly horizon checks whether the vol-clustering consistency property holds beyond the one-day lag. With $T_{\text{lag}}=23$ (vs 27 at $k=1$) the test has lower power, so a higher $p$-value at $k=5$ does not invalidate a $k=1$ PASS. Agreement between $k=1$ and $k=5$ strengthens the pipeline-integrity claim; disagreement is documented but the verdict stays on $k=1$ (pre-pinned primary).

## Verdict — R4-S3-COP consistency check

In [5]:
print('=' * 60)
print('R4-S3-COP CONSISTENCY VERDICT')
print('=' * 60)
print(f'k=1 primary:   α̂₁^COP = {alpha_1_cop_primary:+.6f}, p_1s = {p_one_sided_primary:.6f}  → {verdict_primary}')
print(f'k=5 sensit.:   α̂₁^COP = {alpha_1_cop_sens:+.6f}, p_1s = {p_one_sided_sens:.6f}  → {verdict_sens}')
print()
print(f'VERDICT (k=1, pre-pinned primary): {verdict_primary}')
print()
if verdict_primary == 'CONSISTENCY-FAIL':
    print('HALT — fill disposition memo at')
    print('notebooks/dev_ai_cost_v2/dispositions/<date>-task13-consistency-fail.md')
    print('(suspect data corruption, NOT framework rejection).')
    print()
    print('NOTE: Per v0.2.7 Y-9 closure, cost data is verified at 1.000000× ccusage parity.')
    print('A CONSISTENCY-FAIL CANNOT be parser-divergence — escalate to data-corruption review.')
else:
    print('Pipeline integrity verified — vol propagates correctly through the panel-builder.')
    print('NOT a novel finding (CORRECTIONS-S). NO framework-graduation authority.')
print('=' * 60)

R4-S3-COP CONSISTENCY VERDICT
k=1 primary:   α̂₁^COP = -17.977387, p_1s = 0.670360  → CONSISTENCY-FAIL
k=5 sensit.:   α̂₁^COP = -35.887636, p_1s = 0.873118  → CONSISTENCY-FAIL

VERDICT (k=1, pre-pinned primary): CONSISTENCY-FAIL

HALT — fill disposition memo at
notebooks/dev_ai_cost_v2/dispositions/<date>-task13-consistency-fail.md
(suspect data corruption, NOT framework rejection).

NOTE: Per v0.2.7 Y-9 closure, cost data is verified at 1.000000× ccusage parity.
A CONSISTENCY-FAIL CANNOT be parser-divergence — escalate to data-corruption review.


## Final disclaimers (carried forward)

> **CORRECTIONS-S pinned (repeat):** R4-S3-COP is a **consistency check, NOT a novel finding**. A CONSISTENCY-PASS recovers a well-established USDCOP vol-clustering stylized fact — independent of AI-cost transmission. It confirms that vol propagates correctly through the panel-builder. The arm has **no framework-graduation authority**. The framework-relevant arm is R4-S3-USD (Task 14, spec §2.2.B), which is an independent inferential test with two-sided null on $\alpha_1^{USD}$ — answering a different question (subscription-regime behavioral inelasticity) on a different LHS (`|Δln NotionalCost^USD|`).

> **CORRECTIONS-Y-6 $\hat\pi$ disclaimer (pinned, repeat):** The cost panel applies the 5m-tier rate to total cache-creation tokens; true economic cost is up to $1 + \hat\pi(r_{1h}/r_{5m}-1) \approx 1.40\times$ higher with $\hat\pi \approx 0.399$. The $\hat\alpha_1^{COP}$ coefficient and the one-sided $p$-value reported here are **unaffected** — a constant multiplicative bias on the LHS adds a constant inside the log, which differences away to zero in $\Delta\ln\text{Cost}^{COP}$, and the absolute value of zero is zero. The sign-and-significance verdict is robust to the $\hat\pi$ disclosure.

> **v0.2.7 Y-9 closure (pinned, repeat):** Cost data is verified at **1.000000× ccusage parity** (UTC apples-to-apples). A CONSISTENCY-FAIL fired here would NOT be due to cost-data-pipeline issues or parser-divergence — it would necessarily indicate data corruption downstream of the cost-ingest step and require a disposition memo, NOT a framework rejection.

> **Anti-fishing summary:** LAG_PRIMARY=1, LAG_SENSITIVITY=5, HAC_LAGS=$\lfloor T^{1/3}\rfloor$, ALPHA_LEVEL=0.05 (one-sided), TOKENS_PROXY=`output_tok` — all pinned PRE-DATA in Cell 2. No post-hoc tuning. No switch to two-sided after seeing results. No swap of tokens-proxy. No adjustment of HAC bandwidth. If CONSISTENCY-FAIL fires, the next step is a disposition memo, NOT mechanical continuation to Task 14.